# Introduction

Customer Behaviour and it’s prediction lies 
at the core of every Business Model. From 
Stock Exchange, e-Commerce and 
Automobile to even Presidential Elections, 
predictions serve a great purpose. Most of 
these predictions are based on the data 
available about a person’s activity either 
online or in-person.

## Problem Statement

For the purpose of this Project we will be creating one such Recommendation Engine 
from the ground-up, where every single user, based on there area of interest and ratings, 
would be recommended a list of movies that are best suited for them.


## Dataset Information

1. **ID** –Contains the separate keys for  customer and movies. 
2. **Rating** – A section contains the user ratings for all the movies.
3. **Genre** –Highlights the category of the 
4. **Movie Name** –Name of the movie with respect to the movie id.

## Steps We will take

In this notebook, we will follow a structured approach to wrangle the movie datasets. The steps include:

1. **Data Gathering**: Import necessary libraries (pandas, numpy) and load the datasets.
2. **Data Assessing**: Manually export for review and programmatically check structure, missing values, duplicates, and statistics.
3. **Data Cleaning**: Standardize columns, add property types, merge datasets, handle missing values, convert data types, clean columns, and remove duplicates.
4. **Validation and Merging**: Re-check the cleaned data.
5. **Export Clean Data**: Save the clean dataset for further use.
6. **Exploratory Data Analysis**: Finding the hidden patter inside data distribution.
7. **Model fitting**: fit the data on the best suited model.
8. **Model Evaluation & validation**: check the model accuracy and performance on varius metric
9. **Model Deployement**: Simple flask app to check the model performance

### Data Gathering:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from pathlib import Path

# Split only on the first two commas; remaining commas belong to the movie title.
rows = []
with Path('movie_titles.csv').open(encoding='ISO-8859-1', newline='') as file:
    for line in file:
        fields = line.rstrip('\r\n').split(',', 2)
        if len(fields) == 3:
            rows.append(fields)

df_movies = pd.DataFrame(rows, columns=['movie_id', 'year', 'movie_name'])
df_movies['movie_id'] = pd.to_numeric(df_movies['movie_id'], errors='coerce')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce')

df_movies.head()

,movie_id,year,movie_name
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW


**Note:**  
- There is a Data importing issue in raw data, since there are movies with comma. Hence `movie_name` col is fixed.

In [ ]:
netflix_df = pd.read_csv('combinedNetflixData.txt',  
                        encoding='ISO-8859-1', 
                        header=None, 
                        names=['customer_id', 'rating'], 
                        usecols = [0,1])

In [45]:
netflix_df.head()

,customer_id,rating
0,1:,NaN
1,1488844,3.0
2,822109,5.0
3,885013,4.0
4,30878,4.0


In [47]:
mask = netflix_df["customer_id"].astype(str).str.match(r"^\d+:$")

In [48]:
# Extract movie ID from those rows
netflix_df["movie_id"] = netflix_df["customer_id"].where(mask).str.replace(":", "", regex=False)

In [50]:
netflix_df.head()

,customer_id,rating,movie_id
0,1:,NaN,1
1,1488844,3.0,NaN
2,822109,5.0,NaN
3,885013,4.0,NaN
4,30878,4.0,NaN


In [51]:
# Propagate movie ID to subsequent rows
netflix_df["movie_id"] = netflix_df["movie_id"].ffill()

# Remove the movie-header rows
netflix_df = netflix_df[~mask].copy()

In [52]:
# Convert columns to numeric types
netflix_df["movie_id"] = netflix_df["movie_id"].astype(int)
netflix_df["customer_id"] = pd.to_numeric(netflix_df["customer_id"], errors="coerce").astype("Int64")

print(netflix_df)

          customer_id  rating  movie_id
1             1488844     3.0         1
2              822109     5.0         1
3              885013     4.0         1
4               30878     4.0         1
5              823519     3.0         1
...               ...     ...       ...
24058258      2591364     2.0      4499
24058259      1791000     2.0      4499
24058260       512536     5.0      4499
24058261       988963     3.0      4499
24058262      1704416     3.0      4499

[24053764 rows x 3 columns]


In [36]:
df_movies = df_movies.melt()


In [39]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24058262 entries, 0 to 24058261
Data columns (total 2 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   variable  object
 1   value     object
dtypes: object(2)
memory usage: 367.1+ MB


In [25]:
df_movies.columns = ['movie_id', 'value']

In [40]:
df_movies.head()

,variable,value
0,1:,"1488844,3,2005-09-06"
1,1:,"822109,5,2005-05-13"
2,1:,"885013,4,2005-10-19"
3,1:,"30878,4,2005-12-26"
4,1:,"823519,3,2004-05-03"


array([1])

In [27]:
df_movies['movie_id'] = df_movies['movie_id'].str.extract(r'(\d+)').astype(int)

movie_id
1    24058262
Name: count, dtype: int64

### Data Assessing

- Over Here we are going to perform two type of Data assessing.
1. Mannual Assessing (using Excel)
2. Programatic Assessing (using Pandas or numpy)

- Afterwards we will write all the issues, in order to clean our data.


In [7]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17770 entries, 0 to 17769
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   movie_id    17770 non-null  int64  
 1   year        17763 non-null  float64
 2   movie_name  17770 non-null  object 
dtypes: float64(1), int64(1), object(1)
memory usage: 416.6+ KB


np.int64(4499)

#### Issues with the dataset found

**Data - `movie_titles.csv`**

- **validity issue**: Some movies are having with comma which creating issue. Excel and pandas considering it as new column. Hence importing only 1st column.
- **validity issue**: Some movies name have speacial charecters, need to remove before apply ml.
- **completion issues**: multiple movies has `year` missing. need manual filling.
- **validity issue**: Column are in diffrent format.



**Data - `combinedNetflixData.txt`**

- **Messy Data**: Movie id are given in the first row followed by customer_id, rating and date.
- **validity issue**: columns are found in string format.
- **completion issue**: There is only 4499 movies rating available out of total 17770 movies.



### Data Cleaning

### Validation and Merging

### Export Clean Data

### Exploratory Data Analysis

### Model fitting

### Model Evaluation & validation

### Model Deployement